# English benchmark aggregate — merged English / transfer metrics (paper-facing)

In [1]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd


def resolve_repo_root_static() -> Path:
    cwd = Path.cwd().resolve()
    for root in (cwd, cwd.parent, cwd.parent.parent):
        if (root / "data" / "processed").is_dir() and (root / "notebooks").is_dir():
            return root
    return cwd.parent if (cwd / "notebooks").is_dir() else cwd


REPO_ROOT = resolve_repo_root_static()
RESULTS = REPO_ROOT / "notebooks" / "results"
EN_EVAL = RESULTS / "english_eval"
EN_TRANSFER = RESULTS / "english_transfer_eval"
UNIFIED = RESULTS / "unified_comparison" / "01_unified_models_full_table.csv"
OUT_CSV = EN_EVAL / "english_benchmark_summary.csv"
OUT_MANIFEST = EN_EVAL / "english_benchmark_manifest.json"
EN_EVAL.mkdir(parents=True, exist_ok=True)

DATASETS_ORDER = [
    "english_train",
    "english_val",
    "english_test",
    "english_soft_eval_slice",
    "english_base_eval_slice",
]

print("REPO_ROOT:", REPO_ROOT)
print("english_eval:", EN_EVAL.resolve())
print("english_transfer_eval:", EN_TRANSFER.resolve())


REPO_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
english_eval: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/english_eval
english_transfer_eval: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/english_transfer_eval


In [2]:
def load_metrics_json(path: Path) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as e:
        print("SKIP bad json:", path, e)
        return None


def flatten_one_metrics(path: Path, eval_bundle: str) -> Dict[str, Any]:
    """One row per model run + bundle (english_eval vs english_transfer_eval)."""
    data = load_metrics_json(path)
    if not data:
        return {}
    model_run_id = data.get("model_run_id") or path.parent.name
    per = data.get("per_dataset") or {}
    row: Dict[str, Any] = {
        "model_run_id": model_run_id,
        "eval_bundle": eval_bundle,
        "metrics_json": "",
    }
    try:
        row["metrics_json"] = str(path.relative_to(REPO_ROOT))
    except ValueError:
        row["metrics_json"] = str(path)
    ok_ct = 0
    for ds in DATASETS_ORDER:
        block = per.get(ds)
        if isinstance(block, dict) and block.get("n_evaluated") not in (None, 0):
            row[f"{ds}_accuracy"] = block.get("accuracy")
            row[f"{ds}_macro_f1"] = block.get("macro_f1")
            row[f"{ds}_weighted_f1"] = block.get("weighted_f1")
            row[f"{ds}_n_evaluated"] = block.get("n_evaluated")
            ok_ct += 1
        else:
            row[f"{ds}_accuracy"] = None
            row[f"{ds}_macro_f1"] = None
            row[f"{ds}_weighted_f1"] = None
            row[f"{ds}_n_evaluated"] = None
    row["n_datasets_with_rows"] = ok_ct
    return row


def gather_rows(root: Path, eval_bundle: str) -> List[Dict[str, Any]]:
    rows = []
    if not root.is_dir():
        return rows
    for metrics_path in sorted(root.glob("*/metrics.json")):
        r = flatten_one_metrics(metrics_path, eval_bundle)
        if r:
            rows.append(r)
    return rows


rows_eval = gather_rows(EN_EVAL, "english_eval")
rows_tr = gather_rows(EN_TRANSFER, "english_transfer_eval")
rows_eval = [r for r in rows_eval if r.get("model_run_id") != "60_english_benchmark"]
all_rows = rows_eval + rows_tr
print("Runs found — english_eval:", len(rows_eval), "| english_transfer_eval:", len(rows_tr))
if all_rows:
    df = pd.DataFrame(all_rows)
    print(df[["model_run_id", "eval_bundle", "n_datasets_with_rows"]].to_string(index=False))


Runs found — english_eval: 3 | english_transfer_eval: 1
             model_run_id           eval_bundle  n_datasets_with_rows
      bert_9classes_final          english_eval                     5
           bert_scrubbing          english_eval                     5
label_smoothing_eps01_2ep          english_eval                     5
      bert_9classes_final english_transfer_eval                     5


In [3]:
try:
    from IPython.display import display
except ImportError:
    display = print

# Optional: join Russian-track metrics from unified comparison (same checkpoint folder names)
if all_rows:
    df = pd.DataFrame(all_rows)
    if UNIFIED.exists():
        uni = pd.read_csv(UNIFIED)
        uni_m = uni[uni["track"] == "main"].copy()
        sub = uni_m[["model_name", "display_name", "accuracy", "macro_f1", "worst_gap", "macro_gap"]].rename(
            columns={
                "model_name": "model_run_id",
                "display_name": "display_name_ru",
                "accuracy": "russian_test_accuracy",
                "macro_f1": "russian_test_macro_f1",
                "worst_gap": "russian_worst_gap",
                "macro_gap": "russian_macro_gap",
            }
        )
        df = df.merge(sub, on="model_run_id", how="left")
    else:
        print("No unified table at", UNIFIED, "— skip Russian join.")

    if "english_test_macro_f1" in df.columns and "russian_test_macro_f1" in df.columns:
        df["delta_en_test_macro_f1_minus_ru"] = (
            pd.to_numeric(df["english_test_macro_f1"], errors="coerce")
            - pd.to_numeric(df["russian_test_macro_f1"], errors="coerce")
        )
    df = df.sort_values(["eval_bundle", "model_run_id"], na_position="last")
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_CSV, index=False)
    manifest = {
        "notebook": "english_benchmark_aggregate.ipynb",
        "output_csv": str(OUT_CSV.relative_to(REPO_ROOT)),
        "n_runs": int(len(df)),
        "eval_bundle_counts": {str(k): int(v) for k, v in df["eval_bundle"].value_counts().items()},
        "unified_joined": UNIFIED.exists(),
    }
    OUT_MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Wrote", OUT_CSV)
    print("Wrote", OUT_MANIFEST)
    pd.set_option("display.max_columns", None)
    display(df)
else:
    print("No rows to write.")


Wrote /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/english_eval/english_benchmark_summary.csv
Wrote /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/english_eval/english_benchmark_manifest.json


,model_run_id,eval_bundle,metrics_json,english_train_accuracy,english_train_macro_f1,english_train_weighted_f1,english_train_n_evaluated,english_val_accuracy,english_val_macro_f1,english_val_weighted_f1,english_val_n_evaluated,english_test_accuracy,english_test_macro_f1,english_test_weighted_f1,english_test_n_evaluated,english_soft_eval_slice_accuracy,english_soft_eval_slice_macro_f1,english_soft_eval_slice_weighted_f1,english_soft_eval_slice_n_evaluated,english_base_eval_slice_accuracy,english_base_eval_slice_macro_f1,english_base_eval_slice_weighted_f1,english_base_eval_slice_n_evaluated,n_datasets_with_rows,display_name_ru,russian_test_accuracy,russian_test_macro_f1,russian_worst_gap,russian_macro_gap,delta_en_test_macro_f1_minus_ru
0,bert_9classes_final,english_eval,notebooks/results/english_eval/bert_9classes_f...,0.112146,0.056417,0.078882,2363,0.100592,0.047514,0.064187,507,0.116371,0.055913,0.080885,507,0.122618,0.058448,0.074985,2414,0.142380,0.064050,0.097429,1496,5,Baseline,0.609000,0.621000,0.329,0.116000,-0.565087
1,bert_scrubbing,english_eval,notebooks/results/english_eval/bert_scrubbing/...,0.079560,0.035243,0.049927,2363,0.088757,0.041593,0.055377,507,0.078895,0.037249,0.054918,507,0.085336,0.038864,0.050427,2414,0.045455,0.024487,0.027324,1496,5,Data Scrubbing,0.594192,0.610567,0.300,0.112077,-0.573317
2,label_smoothing_eps01_2ep,english_eval,notebooks/results/english_eval/label_smoothing...,0.212442,0.066437,0.093364,2363,0.209073,0.069552,0.089060,507,0.199211,0.059452,0.079951,507,0.199254,0.064456,0.087487,2414,0.219920,0.047539,0.083723,1496,5,NaN,NaN,NaN,NaN,NaN,NaN
3,bert_9classes_final,english_transfer_eval,notebooks/results/english_transfer_eval/bert_9...,0.112146,0.056417,0.078882,2363,0.100592,0.047514,0.064187,507,0.116371,0.055913,0.080885,507,0.122618,0.058448,0.074985,2414,0.142380,0.064050,0.097429,1496,5,Baseline,0.609000,0.621000,0.329,0.116000,-0.565087


## How to cite this in the thesis / paper

- **Single-language training:** All checkpoints are trained on Russian resume data only.
- **English evaluation** uses aligned ontology labels from notebook **32** (and optional soft slice from **54**); metrics are **not** comparable to Russian accuracy in absolute terms without caveats — the key claim is **relative** ranking across methods and **magnitude of transfer drop** (`delta_en_test_macro_f1_minus_ru`).
- **`english_eval` vs `english_transfer_eval`:** same metric JSON schema; duplicate rows only if you run both bundles for the same `model_run_id` intentionally (e.g. ablation). Usually **59** populates **`english_transfer_eval`** for the main narrative; **e01–e03** populate **`english_eval`** when you evaluate checkpoints living under the English mirror paths.
- **Next gaps:** extend `MODEL_CHECKPOINT_CANDIDATES` in `english_eval_common.py` + add `e*.ipynb stubs` for additional main-line models (GroupDRO, Focal, etc.) so this table covers the full benchmark row set.
